# Phase 4 — Confirmatory run (PREREG v2)

**When do conformal hallucination guarantees survive distribution shift?**
Code is pinned to the git tag **`prereg-v2`** (frozen before any test data exists).

| What | Data | Models |
|---|---|---|
| Calibration | COCO Cal-A + Cal-B (700 images) | LLaVA-1.5-7B, OWLv2 |
| Test (locked) | COCO test split (500) + 804 AMBER images | LLaVA, Qwen2.5-VL-3B, InternVL3-2B; OWLv2, OWL-ViT v1, Grounding DINO; 25 corruptions; object removal |

**Kaggle settings:** Accelerator **GPU T4 x2**, **Internet ON**, Persistence: Files only (optional).

**How to run**
1. Run cells 1–5 interactively (setup + smoke test, ~25 min). Cell 5 must print captions from all three VLMs and some detector scores.
2. **Save Version → Save & Run All (Commit)**. Both GPUs work in parallel (~8–11 h). You can close the browser.
3. If the version stops at the 12 h limit: open a new notebook version, **Add Input → Your Work → (this notebook's previous version)**, and Save & Run All again. Cell 4 imports the finished work and only the rest is computed.
4. Download `BCRBP_phase4/phase4_results.zip` from the version's Output tab and put it in Drive → `BCRBP_phase2`.

In [ ]:
# 1. Environment and GPUs
import os, subprocess, sys, time
ON_KAGGLE = os.path.exists("/kaggle/working")
print(subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
      or "NO GPU FOUND - switch the accelerator to GPU T4 x2")
import torch
NGPU = torch.cuda.device_count()
print("GPUs:", NGPU)

In [ ]:
# 2. Folders
WORK = "/kaggle/working" if ON_KAGGLE else os.getcwd()
OUT = os.path.join(WORK, "BCRBP_phase4")
SMOKE = os.path.join(WORK, "BCRBP_phase4_smoke")
os.makedirs(OUT, exist_ok=True)
print("Results go to:", OUT)

In [ ]:
# 3. Code at the pre-registered tag, and packages
REPO = "https://github.com/antonyjasfer/Budget-Coupled-Robust-Belief-Propagation-for-Visual-Hallucination-Detection.git"
TAG = "prereg-v2"
os.chdir(WORK)
if not os.path.exists("bcrbp"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", TAG, REPO, "bcrbp"], check=True)
os.chdir(os.path.join(WORK, "bcrbp"))
!pip -q install inflect bitsandbytes accelerate gdown
import transformers
print("torch", torch.__version__, "| transformers", transformers.__version__)
!git log --oneline -1

In [ ]:
# 4. Setup, images, and import of any earlier session's work (CPU, ~10-15 min)
!python scripts/phase4.py setup  --out "$OUT"
!python scripts/phase4.py import --out "$OUT"
!python scripts/phase4.py images --out "$OUT"

In [ ]:
# 5. Smoke test in a separate folder (2 images per set, ~10-15 min)
import shutil, json, glob
os.makedirs(os.path.join(SMOKE, "amber"), exist_ok=True)
for f in ["synonyms.txt", "prep.json"]:
    shutil.copy(os.path.join(OUT, f), os.path.join(SMOKE, f))
shutil.copy(os.path.join(OUT, "amber", "annotations.json"), os.path.join(SMOKE, "amber", "annotations.json"))
for src, dst in [(os.path.join(OUT, "images"), os.path.join(SMOKE, "images")),
                 (os.path.join(OUT, "amber", "images"), os.path.join(SMOKE, "amber", "images"))]:
    if not os.path.exists(dst):
        os.symlink(src, dst)
!python scripts/phase4.py gpu --out "$SMOKE" --limit 2
show = lambda stem: [json.loads(l) for p in glob.glob(f"{SMOKE}/{stem}.s*.jsonl") for l in open(p) if l.strip()]
for stem in ["captions_llava_coco", "captions_llava_amber", "captions_qwen", "captions_internvl"]:
    rows = show(stem)
    print(f"\n{stem}: {len(rows)} rows", "|", (rows[0]["caption"][:200] if rows else "MISSING"))
print("\nowl rows:", len(show("owl")), "| swap rows:", len(show("swap")))

In [ ]:
# 6. Full run: one process per GPU (~8-11 h). Logs: OUT/gpu_<k>.log
procs = []
for k in range(max(1, NGPU)):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(k))
    log = open(os.path.join(OUT, f"gpu_{k}.log"), "a")
    procs.append(subprocess.Popen([sys.executable, "scripts/phase4.py", "gpu", "--out", OUT,
                                   "--shard", str(k), "--nshards", str(max(1, NGPU))],
                                  stdout=log, stderr=subprocess.STDOUT, env=env))
while any(p.poll() is None for p in procs):
    time.sleep(600)
    for k in range(len(procs)):
        tail = open(os.path.join(OUT, f"gpu_{k}.log")).read().splitlines()[-1:]
        print(time.strftime("%H:%M"), f"GPU {k}:", tail[0] if tail else "")
print("exit codes:", [p.returncode for p in procs])
for k in range(len(procs)):
    if procs[k].returncode:
        print(f"--- GPU {k} log tail ---"); print("\n".join(open(os.path.join(OUT, f'gpu_{k}.log')).read().splitlines()[-30:]))

In [ ]:
# 7. Confirmatory analysis (CPU, ~20-40 min) and report
!python scripts/phase4.py analyze --out "$OUT"
from IPython.display import Markdown, display
display(Markdown(open(f"{OUT}/phase4_report.md").read()))

In [ ]:
# 8. Package results (no images)
import zipfile
zpath = os.path.join(OUT, "phase4_results.zip")
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for p in glob.glob(f"{OUT}/*.jsonl") + glob.glob(f"{OUT}/*.log") + [f"{OUT}/prep.json", f"{OUT}/phase4_results.json", f"{OUT}/phase4_report.md"]:
        if os.path.exists(p):
            z.write(p, os.path.basename(p))
    z.writestr("environment.txt", f"torch {torch.__version__}\ntransformers {transformers.__version__}\n"
               + subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout)
print("Saved:", zpath, round(os.path.getsize(zpath) / 1e6, 1), "MB")